# Lab 11: Zero-Bus Streaming with Zerobus

Implementing an event-driven ingestion pipeline **without a centralized message broker** using Databricks Zerobus (push-based single-sink model). The pipeline streams user movie consumption events — `movie_started`, `watch_progress`, `rating_submitted`, `watchlist_added` — into `main.lab_data.movie_stream_events` (Delta table with Change Data Feed enabled) and guarantees **idempotent ingestion** to handle network retries and at-least-once delivery.

**Notebook:** `01_zerobus_producer_and_idempotency.ipynb`

In [0]:
CATALOG = "main"
SCHEMA = "lab_data"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Active context: {CATALOG}.{SCHEMA}")

## Setup — Delta Table with Change Data Feed

Create the target Delta table `movie_stream_events` with **Change Data Feed (CDF)** enabled. CDF tracks row-level changes (inserts, updates, deletes) which is essential for downstream CDC consumers and audit trails in a Zero-Bus architecture.

In [0]:
def create_movie_stream_events_table():
    """Create the movie_stream_events Delta table with Change Data Feed enabled."""
    spark.sql(f"""
        CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.movie_stream_events (
            event_id STRING,
            user_id STRING,
            movie_title STRING,
            event_type STRING,
            watch_time_seconds INT,
            event_timestamp TIMESTAMP,
            ingested_at TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.enableChangeDataFeed' = 'true'
        )
    """)
    print("Table 'main.lab_data.movie_stream_events' was created successfully.")


create_movie_stream_events_table()

## Step 1 — Event Producer with Intentional Duplicates

Generate synthetic movie-streaming events and deliberately duplicate the first two `event_id` values to simulate **at-least-once delivery** — the real-world scenario where a producer re-sends events it didn't receive an ACK for due to network instability.

In [0]:
import uuid
import random
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

def generate_movie_events(num_events: int = 10, inject_duplicates: bool = True) -> list:
    """Generate synthetic movie-streaming events for the ZeroBus producer."""
    movie_titles = ["Inception", "The Dark Knight", "Interstellar", "Avatar", "Titanic"]
    event_types = ["movie_started", "watch_progress", "rating_submitted", "watchlist_added"]

    events = []
    for _ in range(num_events):
        events.append({
            "event_id": str(uuid.uuid4()),
            "user_id": f"user_{random.randint(100, 105)}",
            "movie_title": random.choice(movie_titles),
            "event_type": random.choice(event_types),
            "watch_time_seconds": random.randint(30, 7200),
            "event_timestamp": datetime.now(timezone.utc)
        })

    # Deliberately duplicate the first two events to stress-test idempotency
    if inject_duplicates and len(events) >= 2:
        events.append(events[0].copy())
        events.append(events[1].copy())
        print(f"⚠️  Generated {len(events)} events (including 2 intentional duplicates).")
    else:
        print(f"✅ Generated {len(events)} unique events.")

    return events

# Generate a test batch with duplicates
batch_events = generate_movie_events(num_events=5, inject_duplicates=True)

## Step 2 — Idempotent Ingestion via MERGE INTO

Implement **two-tier deduplication** to guarantee exactly-once semantics over at-least-once delivery:

1. **Intra-batch dedup:** `dropDuplicates(["event_id"])` removes duplicates within the same batch before they reach the target table.
2. **Inter-batch idempotency:** `MERGE INTO ... ON target.event_id = source.event_id WHEN NOT MATCHED THEN INSERT` ensures that replayed events from prior batches are silently skipped — the table state converges to the same result regardless of how many times a batch is re-sent.

In [0]:
from pyspark.sql import functions as F

def ingest_events_idempotent(
    events_data: list,
    target_table: str = f"{CATALOG}.{SCHEMA}.movie_stream_events"
) -> None:
    """
    Idempotent ingestion of events into a Delta table.
    If event_id already exists, the record is skipped (deduplication via MERGE).
    """
    schema = StructType([
        StructField("event_id", StringType(), False),
        StructField("user_id", StringType(), True),
        StructField("movie_title", StringType(), True),
        StructField("event_type", StringType(), True),
        StructField("watch_time_seconds", IntegerType(), True),
        StructField("event_timestamp", TimestampType(), True)
    ])

    # 1. Convert the event list to a Spark DataFrame and add ingestion timestamp
    incoming_df = (spark.createDataFrame(events_data, schema=schema)
                       .withColumn("ingested_at", F.current_timestamp()))

    # 2. Drop intra-batch duplicates before writing
    deduped_incoming = incoming_df.dropDuplicates(["event_id"])
    deduped_incoming.createOrReplaceTempView("staged_incoming_events")

    # 3. MERGE INTO: insert only events not already present in the target table
    spark.sql(f"""
        MERGE INTO {target_table} AS target
        USING staged_incoming_events AS source
          ON target.event_id = source.event_id
        WHEN NOT MATCHED THEN
          INSERT (event_id, user_id, movie_title, event_type,
                  watch_time_seconds, event_timestamp, ingested_at)
          VALUES (source.event_id, source.user_id, source.movie_title, source.event_type,
                  source.watch_time_seconds, source.event_timestamp, source.ingested_at)
    """)

    total_count = spark.table(target_table).count()
    print(f"📥 Batch processed successfully. Total unique records in table: {total_count}")

# Perform the first ingestion
ingest_events_idempotent(batch_events)

## Step 3 — Idempotency Test (Network Retry Simulation)

Re-send the **exact same batch** of events to verify that idempotent ingestion prevents duplicate rows. The table count should remain unchanged after the retry, proving the system handles network retries gracefully without producing duplicates.

In [0]:
# --- Idempotency test: re-send the exact same batch (simulates a network retry) ---
print("--- 🔁 Re-sending THE SAME batch (simulating Network Retry) ---")
ingest_events_idempotent(batch_events)

# Verify final table contents — should still have only the unique events
print("\n📋 Final table contents:")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.movie_stream_events ORDER BY ingested_at"))

In [0]:
# ─────────────────────────────────────────────────────────────────────
#  Architecture Comparison: Traditional Kafka vs Databricks Zero-Bus
# ─────────────────────────────────────────────────────────────────────
#  This cell provides a side-by-side visual comparison of two streaming
#  architectures, covering infrastructure, cost, operational overhead,
#  and scalability — then estimates monthly TCO for both.
# ─────────────────────────────────────────────────────────────────────

def print_comparison_table():
    """Print a visual side-by-side comparison of Kafka vs Zero-Bus."""
    rows = [
        ("Category",              "Traditional Kafka",                      "Databricks Zero-Bus"),
        ("────────────────────────", "──────────────────────────────────", "──────────────────────────────────"),
        ("Message Broker",        "Dedicated cluster (3+ brokers)",         "None (push-based, no broker)"),
        ("Compute Model",         "24/7 running Spark/Consumer",            "Serverless, pay-per-ingested-GB"),
        ("Scaling",               "Manual partition & broker scaling",      "Auto-scales with load"),
        ("Operational Overhead",  "High (manage, patch, monitor brokers)",  "Minimal (managed service)"),
        ("Fault Tolerance",       "Broker replication factor tuning",      "Built-in retry + idempotent MERGE"),
        ("Delivery Semantics",    "At-least-once (requires app logic)",    "Exactly-once via MERGE + CDF"),
        ("Idle Cost",             "Always-on compute + broker infrastructure", "~$0 when idle (serverless)"),
        ("Time to Deploy",        "Days (provision, configure, test)",       "Minutes (CREATE TABLE + MERGE)"),
        ("Best For",              "Multi-consumer, high-throughput pub/sub", "Single-sink, event-driven ingestion"),
    ]

    col_widths = [24, 40, 40]
    sep = "│"

    # Header row
    header = sep.join(f" {r:<{w-2}} " for r, w in zip(rows[0], col_widths))
    print(header)
    print("─" * len(header))

    # Data rows
    for row in rows[1:]:
        print(sep.join(f" {cell:<{w-2}} " for cell, w in zip(row, col_widths)))


def estimate_monthly_cost(events_per_day=5_000_000):
    """Estimate and visually compare monthly TCO for both architectures."""
    # Kafka: 3 brokers (m5.large) + 24/7 single-node Spark Streaming
    kafka_infra = (3 * 0.10 * 24 * 30) + (0.40 * 24 * 30)  # ~$500/міс
    # Zero-Bus: pay per ingested GB (serverless)
    gb_per_month = (events_per_day * 500 / 1e9) * 30  # ~75 GB
    zerobus_cost = gb_per_month * 0.05 + 10  # ~$15/міс

    kafka_pct = kafka_infra / (kafka_infra + zerobus_cost) * 100
    zerobus_pct = 100 - kafka_pct

    print("\n" + "=" * 78)
    print("  📊  ESTIMATED MONTHLY TCO COMPARISON")
    print("=" * 78)
    print(f"  • Traditional Kafka + 24/7 Compute : ~${kafka_infra:.2f}  ({kafka_pct:.0f}% of total)")
    print(f"  • Databricks Zero-Bus (Serverless)  : ~${zerobus_cost:.2f}  ({zerobus_pct:.0f}% of total)")
    print("=" * 78)

    # Simple ASCII bar chart
    max_bar = 50
    kafka_bar_len = int(kafka_pct / 100 * max_bar)
    zerobus_bar_len = max_bar - kafka_bar_len

    print(f"\n  Kafka     [{'█' * kafka_bar_len:<{max_bar}}] ${kafka_infra:.2f}")
    print(f"  Zero-Bus  [{'█' * zerobus_bar_len:<{max_bar}}] ${zerobus_cost:.2f}")

    savings = kafka_infra - zerobus_cost
    savings_pct = (savings / kafka_infra) * 100
    print(f"\n  💰  Estimated savings with Zero-Bus: ~${savings:.2f}/mo  ({savings_pct:.0f}% reduction)")
    print("=" * 78 + "\n")


print_comparison_table()
estimate_monthly_cost()